# Trees chapter project - the wine-quality running score

Build one classifier **three ways** on the same train/test split and keep a **running
scoreboard**: a single tree (HW1) -> a random forest (HW2) -> gradient boosting (HW3).
Report accuracy + **F1 / ROC-AUC** each time and watch how - and *whether* - each ensemble
beats the last.

> The lectures use **Titanic**, where one feature (`gender`) dominates, so the ensembles
> barely help. Here you apply the same pipeline to a **richer** dataset - **wine quality**
> (~6,500 wines, 12 numeric features, no single dominant one) - where the ensembles have
> real room to win.

**Task:** predict whether a wine is **"good"** (quality score >= 7). About **20%** are good,
so the data is **imbalanced** - always predicting "not good" already scores ~0.80 accuracy.
**Report ROC-AUC and F1, not just accuracy.**

Data is pinned in `data/wine.csv` (UCI wine quality, red + white combined), so nothing is
fetched at run time.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

SEED = 509

df = pd.read_csv('data/wine.csv')
y = (df['quality'] >= 7).astype(int)          # 'good' wine = quality 7 or above
X = df.drop(columns=['quality'])              # 12 numeric features (11 chemistry + is_red)
feature_names = list(X.columns)

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=SEED, stratify=y)
print(X_tr.shape, X_te.shape, '| good(>=7) rate:', round(float(y.mean()), 3),
      '| majority-baseline acc:', round(float(1 - y.mean()), 3))

In [ ]:
scores = {}   # name -> {'acc':..., 'f1':..., 'auc':...}

def record(name, model):
    """Fit on train, score on test, add a row to the running scoreboard."""
    model.fit(X_tr, y_tr)
    pred = model.predict(X_te)
    proba = model.predict_proba(X_te)[:, 1]
    scores[name] = dict(acc=accuracy_score(y_te, pred),
                        f1=f1_score(y_te, pred),
                        auc=roc_auc_score(y_te, proba))
    print(f"{name:22s} acc={scores[name]['acc']:.3f}  "
          f"f1={scores[name]['f1']:.3f}  auc={scores[name]['auc']:.3f}")
    return model

## HW1 - grow, prune, read a tree ([17])

1. **By hand** (Play Tennis, on paper): build the stump on Outlook, compute the information
   gain of one other split.
2. **Fit** a `DecisionTreeClassifier`; plot the depth train/test **U-curve** (AUC vs `max_depth`).
3. **Prune** via `ccp_alpha` chosen by cross-validation; visualize the final tree (`plot_tree`).
4. `record('pruned tree', ...)` it onto the scoreboard.

*Bonus:* confirm **scale-invariance** (rescale `alcohol`, refit, tree unchanged); refit the tree
as `LGBMClassifier(n_estimators=1, learning_rate=1.0)` and check it matches.

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree

# TODO: AUC-vs-max_depth U-curve; pick ccp_alpha by CV; visualize with plot_tree; then record().
# tree = record('pruned tree', DecisionTreeClassifier(ccp_alpha=..., random_state=SEED))

## HW2 - bag the tree ([18])

1. Fit a `RandomForestClassifier(oob_score=True)`; compare the **OOB score** to **5-fold CV**;
   sweep `n_estimators` (watch the **plateau**) and `max_features` (watch it matter).
   `record('random forest', ...)` it - and compare its AUC/F1 to your HW1 tree.
2. **The importance trap** (next cell): can you trust the default importance bar chart?
3. **Decision boundaries** (cell after): watch the tree's staircase smooth out.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold

# TODO: fit RF(oob_score=True); compare rf.oob_score_ vs 5-fold CV
#       (use StratifiedKFold(shuffle=True, random_state=SEED));
#       sweep n_estimators + max_features; then record('random forest', rf).
# rf = record('random forest', RandomForestClassifier(n_estimators=300, oob_score=True,
#                                                      random_state=SEED, n_jobs=-1))

### The importance trap - which importance can you trust?

`rf.feature_importances_` (the default) is **impurity importance**, and it is **biased toward
features with many split points** (continuous / high-cardinality ones). To expose it, add two
columns of **pure junk** and see whether the model is fooled.

In [ ]:
from sklearn.inspection import permutation_importance
rng = np.random.default_rng(SEED)

X_tr_j, X_te_j = X_tr.copy(), X_te.copy()
X_tr_j['rand_noise']   = rng.normal(size=len(X_tr_j))                      # continuous noise
X_te_j['rand_noise']   = rng.normal(size=len(X_te_j))
X_tr_j['rand_hi_card'] = rng.integers(0, 4000, len(X_tr_j)).astype(float)  # random high-cardinality id
X_te_j['rand_hi_card'] = rng.integers(0, 4000, len(X_te_j)).astype(float)

rf_j = RandomForestClassifier(n_estimators=300, random_state=SEED,
                              n_jobs=-1).fit(X_tr_j, y_tr)

# TODO: put the two rankings side by side, sorted:
#   (a) impurity   : rf_j.feature_importances_                          (learned from TRAIN)
#   (b) permutation: permutation_importance(rf_j, X_te_j, y_te,
#                        n_repeats=10, random_state=SEED).importances_mean   (from TEST)
# Q1: how much impurity importance do 'rand_noise' / 'rand_hi_card' get, and where does the
#     real 'is_red' feature rank by impurity? (You should find the junk scoring ABOVE is_red.)
# Q2: where do the two junk columns land on PERMUTATION importance? Explain the gap. ([18])

### Decision boundaries - blocky tree -> smooth forest

Pick **two** features and look at what each model draws. The single tree makes the blocky,
axis-aligned **staircase** from [17]; averaging many trees ([18]) rounds it off; boosting
carves it differently again. A helper is provided - you do the fitting.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.ensemble import HistGradientBoostingClassifier

def plot_regions(model, ax, X2, y2, title):
    """Shade P(good) over a 2-feature grid, training points on top."""
    x0 = np.linspace(X2.iloc[:, 0].min() - 0.3, X2.iloc[:, 0].max() + 0.3, 300)
    x1 = np.linspace(X2.iloc[:, 1].min() - 0.3, X2.iloc[:, 1].max() + 0.3, 300)
    xx, yy = np.meshgrid(x0, x1)
    grid = pd.DataFrame(np.c_[xx.ravel(), yy.ravel()], columns=X2.columns)
    zz = model.predict_proba(grid)[:, 1].reshape(xx.shape)
    ax.contourf(xx, yy, zz, levels=20, cmap='coolwarm', alpha=0.7)
    ax.scatter(X2.iloc[:, 0], X2.iloc[:, 1], c=y2, cmap='coolwarm', s=6,
               edgecolor='k', linewidths=0.2)
    ax.set_title(title); ax.set_xlabel(X2.columns[0]); ax.set_ylabel(X2.columns[1])

# TODO: feats2 = ['alcohol', 'volatile_acidity']; fit a single tree / RF / GBDT on X_tr[feats2],
#       make 3 side-by-side axes (plt.subplots(1, 3)) and call plot_regions for each.
#       Compare: blocky staircase ([17]) vs smoother RF ([18]) vs boosting.

## HW3 - boost the tree ([19]-[20])

1. **By hand** (Yerevan rent toy): do **3 rounds** of gradient boosting - init = mean(y),
   compute residuals, fit a stump, update with eta. Show the residuals shrinking.
2. Fit a `HistGradientBoostingClassifier` with **early stopping**; `record(...)` it and compare
   to your HW2 forest (accuracy + F1 / ROC-AUC).
3. Tune `learning_rate`; then try **LightGBM** with a **monotonic constraint** on a feature whose
   effect you expect to be monotone (e.g. `alcohol` - check the direction first).
4. **Threshold tuning ([13]).** The scoreboard uses the default 0.5 cutoff; only ~20% of wines
   are "good", so pick the **F1-optimal threshold out-of-fold** (never on test) and re-report F1.

*Bonus:* force overfitting (train -> perfect, test U-turn) then early-stop; try `subsample < 1`.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier

# TODO: HistGradientBoosting(early_stopping=True) -> record(); then lightgbm.LGBMClassifier
#       with monotone_constraints. Compare to your HW2 forest on the scoreboard.
# gb = record('hist gradient boosting', HistGradientBoostingClassifier(random_state=SEED))

## The running scoreboard

Unlike Titanic in the lectures, here the ensembles have real room to win: a single tree lands
around **AUC 0.74**, while the random forest and gradient boosting reach **~0.92**. Does your
scoreboard show the same jump? Which ensemble edges ahead - and does tuning ([20]) change the
ranking?

In [ ]:
pd.DataFrame(scores).T.round(3)